In [63]:
import numpy as np 
import faiss
import sys
import time
import csv
import os
module_path = '/home/cpanourg/projects/2-hdvc/'

if module_path not in sys.path:
    sys.path.append(module_path)

from src.utils import read_fvecs
from src.utils import append_or_create_csv

In [64]:
# Loading the GIST dataset
db = np.array(read_fvecs('/data/cpanourg/2-hdvc/data/gist/gist_base.fvecs'))
qr = np.array(read_fvecs('/data/cpanourg/2-hdvc/data/gist/gist_query.fvecs'))


Reading File - /data/cpanourg/2-hdvc/data/gist/gist_base.fvecs:(1000000, 960)
Reading File - /data/cpanourg/2-hdvc/data/gist/gist_query.fvecs:(1000, 960)


In [119]:
dataset_name = 'GIST'
sampling_method = 'random'  # 'random'
train_size_ratio = 0.1  # Ratio of the database to use for training if using sampling
nb = db.shape[0]  # Number of database vectors
nq = qr.shape[0]  # Number of query vectors
k = 100 # Number of nearest neighbors to search for
dim = db.shape[1]  # Dimensionality of the vectors

nbits = 8 # 2^nbits is the number of centroids for each subquantizer
n_subquantizers = 8   # Number of subquantizers for the PQ

In [120]:

start = time.time()
# Step 2: Initialize the product quantizer
pq = faiss.IndexPQ(dim, n_subquantizers, nbits)

# nlist = 100  # Number of Voronoi cells (clusters)
# quantizer = faiss.IndexFlatL2(dim)  # the other index
# pq = faiss.IndexIVFPQ(quantizer, dim, nlist,  n_subquantizers, nbits)

# Step 3: Train the PQ on the database (training data is typically a random sample of the DB)
# If you have a separate training set, you should use that instead of the full database.
if sampling_method == 'random':
    num_samples = min(int(train_size_ratio * nb), db.shape[0])  # Number of samples for training
    training_data = db[np.random.choice(db.shape[0], num_samples, replace=False)]
    print(f"Training PQ on {num_samples} random samples from the database.")

# Perform PQ training
pq.train(training_data)

# Step 4: Add the database vectors to the PQ index
pq.add(db)
end = time.time()
pq_train_add_time = np.round(end - start, 2)
print(f"Time to train and add to PQ index: {pq_train_add_time} seconds")
print("is_trained:", pq.is_trained)


start = time.time()
# Step 7: Compute approximate PQ distances
D_pq, I_pq = pq.search(qr, k)  # Approximate distances (PQ) to all DB vectors
end = time.time()   
pq_dist_time = np.round(end - start, 2)
print(f"Time to compute PQ distances: {pq_dist_time} seconds")



Training PQ on 100000 random samples from the database.
Time to train and add to PQ index: 4.19 seconds
is_trained: True
Time to compute PQ distances: 0.21 seconds


In [ ]:
# Copy the codes into a NumPy array
codes_array = faiss.vector_to_array(pq.codes)  # returns a new numpy.ndarray
N = pq.ntotal        # total number of vectors in the index
code_size = pq.code_size  # bytes per vector code (product of subquantizers * bits/8, rounded up):contentReference[oaicite:6]{index=6}

codes_matrix = codes_array.reshape(N, code_size)
print(codes_matrix.shape)  # should be (N, code_size)
codes_matrix

(1000000, 8)


array([[ 33,  58,  59, ..., 194, 188,  80],
       [146, 226, 174, ..., 174,  18, 155],
       [ 69,  66, 116, ...,  14,  26,  10],
       ...,
       [  3, 100, 122, ...,   3, 203,  40],
       [243,  95, 140, ...,  89,  11,  11],
       [210, 212, 179, ...,  73, 110, 210]], dtype=uint8)

In [122]:

start = time.time()
# Step 5: Calculate exact Euclidean distances using the flat index
index_flat = faiss.IndexFlatL2(dim)  # Exact L2 index
index_flat.add(db)
end = time.time()

l2_add_time = np.round(end - start, 2)
print(f"Time to add to Flat index: {l2_add_time} seconds")

start = time.time()
# Step 6: Compute the distances (all distances between query and database)
D_exact, I_exact = index_flat.search(qr, k)  # Exact distances (Euclidean) to all DB vectors
end = time.time()

l2_dist_time = np.round(end - start, 2)
print(f"Time to compute exact distances: {l2_dist_time} seconds")


Time to add to Flat index: 1.96 seconds
Time to compute exact distances: 2.01 seconds


In [123]:
# # Step 8: Compute relative error matrix
# # Define a small epsilon value to replace zero distances
# epsilon = 1e-6

# # Replace zeros in D_exact with epsilon
# D_exact_safe = np.where(D_exact == 0, epsilon, D_exact)

# # Now compute the relative error
# rel_error_matrix = np.abs(D_exact_safe - D_pq) / D_exact_safe

# rel_error_mean = np.mean(rel_error_matrix)
# rel_error_std = np.std(rel_error_matrix)


In [124]:
def recall_at_k(I_true, I_test, k):
    n_queries = I_true.shape[0]
    recall = 0
    for i in range(n_queries):
        # True neighbors (I_true[i]) should be compared against the test neighbors (I_test[i])
        # Ensure you're comparing sets of indices in the top k results
        recall += len(set(I_true[i, :k]) & set(I_test[i, :k])) / k
    return recall / n_queries


recallk = recall_at_k(I_exact, I_pq, k)

print(f"Recall@{k}: {recallk:.4f}")


Recall@100: 0.1034


In [126]:


# Example usage
header = [
    'method', 'dataset',
    # 'mean_rel_error', 'std_rel_error',
    f'recall@{k}',
    'qr_size', 'db_size', 'train_ratio', 'sample_method',
    'train_add_time', 'dist_time', 'subspaces', 'centroids', 'bits'
]

rows_to_append = [[
    'PQ', dataset_name,
    # rel_error_mean, rel_error_std,
    recallk,
    nq, nb, train_size_ratio, sampling_method,
    pq_train_add_time, pq_dist_time,
    n_subquantizers, 2**nbits, nbits
]]

In [127]:
# Specify the CSV file name
file_name = '/data/cpanourg/2-hdvc/results/rel_error.csv'

# Call the function to append or create the file
append_or_create_csv(file_name, header, rows_to_append)


In [130]:
import pandas as pd
pd.read_csv('/data/cpanourg/2-hdvc/results/rel_error.csv').round(2)


,method,dataset,recall@100,qr_size,db_size,train_ratio,sample_method,train_add_time,dist_time,subspaces,centroids,bits
0,PQ,GIST,0.10,1000,1000000,0.1,random,4.19,0.21,8,256,8
1,OPQ,GIST,0.24,1000,1000000,0.1,random,56.96,0.17,8,256,8
